# HealthGuard AI — Phase 4: CNN vs LSTM on MIT-BIH ECG Arrhythmia

**Module:** 6WCM0029 Final Year Project
**Author:** Muhammad Zubair, University of Hertfordshire

## Purpose
The tabular phase (8 models across UCI Heart Disease and Pima Diabetes) established that
gradient-boosting and pretrained transformer models outperform deep learning on small,
single-snapshot tabular records. This notebook tests the opposite regime: **sequential
biosignal data**, where deep learning is expected to win.

This directly answers supervisor feedback point (15) — *"look into RNN/LSTM"* — and
provides the justified architectural comparison required for the viva.

## Models compared
| Model | Inductive bias | Hypothesis |
|---|---|---|
| **1D CNN** | Local morphology (QRS shape, P/T wave form) | Strong — arrhythmia class is largely a *shape* problem |
| **BiLSTM** | Long-range temporal dependency across the beat | Weaker — 187 samples of a single beat has limited long-range structure |
| **CNN-BiLSTM hybrid** | Local features then temporal context | Best of both, at higher cost |

## Headline metric: **macro F1**, not accuracy
The dataset is ~82.8% normal beats. A model predicting "Normal" for everything scores 82.8%
accuracy and is clinically useless. Macro F1 weights all five classes equally and is the
honest metric here. This is a deliberate, defensible evaluation decision — raise it in the viva.

---
## 1. Environment check
Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)

import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs visible:", gpus)
assert gpus, "No GPU! Runtime > Change runtime type > T4 GPU, then re-run."

In [ ]:
import os, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score, roc_auc_score,
                             precision_recall_fscore_support)

from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 110
print("Imports OK")

---
## 2. Dataset

**Source:** `shayanfazeli/heartbeat` on Kaggle — the standard preprocessed MIT-BIH
Arrhythmia release. Beats are segmented, resampled to 125 Hz, zero-padded to a fixed
**187-sample** window, and amplitude-normalised to [0, 1].

- `mitbih_train.csv` — 87,554 beats
- `mitbih_test.csv` — 21,892 beats
- **Total: 109,446 beats**, 187 features + 1 label column (no header row)

### AAMI class mapping
| Code | AAMI class | Meaning |
|---|---|---|
| 0 | N | Normal / bundle branch block |
| 1 | S | Supraventricular ectopic |
| 2 | V | Ventricular ectopic |
| 3 | F | Fusion of ventricular and normal |
| 4 | Q | Unclassifiable / paced |

### Getting your Kaggle token
kaggle.com → your avatar → Settings → API → **Create New Token** → downloads `kaggle.json`.
Run the cell below and upload it.

In [ ]:
from google.colab import files
import os

os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Upload your kaggle.json:")
    up = files.upload()
    with open('/root/.kaggle/kaggle.json','wb') as f:
        f.write(list(up.values())[0])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kaggle
!kaggle datasets download -d shayanfazeli/heartbeat -p /content/data --unzip
!ls -lh /content/data

In [ ]:
DATA = '/content/data'
train_df = pd.read_csv(f'{DATA}/mitbih_train.csv', header=None)
test_df  = pd.read_csv(f'{DATA}/mitbih_test.csv',  header=None)

print("train:", train_df.shape, " test:", test_df.shape,
      " total:", len(train_df)+len(test_df))

CLASS_NAMES = ['N (Normal)', 'S (Supravent.)', 'V (Ventricular)',
               'F (Fusion)', 'Q (Unclassified)']
SHORT = ['N','S','V','F','Q']
N_CLASSES = 5

dist = train_df[187].value_counts().sort_index()
summary = pd.DataFrame({
    'class': SHORT,
    'train_n': dist.values,
    'train_%': (100*dist/dist.sum()).round(2).values,
    'test_n': test_df[187].value_counts().sort_index().values
})
display(summary)
print(f"\nImbalance ratio (majority:minority) = {dist.max()/dist.min():.1f} : 1")

### Signal morphology — one beat per class
Worth putting in the report: it shows *visually* why a convolutional filter bank is the
right prior. The classes differ by waveform **shape**, not by long-range temporal structure.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.2), sharey=True)
for c in range(N_CLASSES):
    sub = train_df[train_df[187] == c].iloc[:, :187]
    for i in range(min(8, len(sub))):
        axes[c].plot(sub.iloc[i].values, lw=0.7, alpha=0.5, color='steelblue')
    axes[c].plot(sub.mean().values, lw=2.2, color='crimson', label='class mean')
    axes[c].set_title(CLASS_NAMES[c], fontsize=10)
    axes[c].set_xlabel('sample (125 Hz)')
    axes[c].legend(fontsize=7)
axes[0].set_ylabel('normalised amplitude')
plt.suptitle('MIT-BIH beat morphology by AAMI class', y=1.06, fontsize=13)
plt.tight_layout(); plt.savefig('/content/fig_morphology.png', bbox_inches='tight'); plt.show()

---
## 3. Preprocessing

**Decisions made here (justify these in the report):**

1. **Keep the published train/test split.** Every benchmark paper on this release uses it —
   changing it makes your numbers incomparable to the literature.
2. **Stratified 10% validation carved from train**, used only for early stopping. The test
   set is touched exactly once, at the end.
3. **Balanced class weights instead of SMOTE.** Synthetic oversampling of ECG waveforms
   risks fabricating physiologically impossible beats. Loss reweighting achieves the same
   goal without inventing data.
4. **No further normalisation** — the source release is already amplitude-normalised.
5. Input reshaped to `(n, 187, 1)` — one channel, 187 timesteps.

In [ ]:
X = train_df.iloc[:, :187].values.astype('float32')
y = train_df[187].values.astype('int32')
X_test = test_df.iloc[:, :187].values.astype('float32')
y_test = test_df[187].values.astype('int32')

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=SEED)

# (n, timesteps, channels)
X_tr   = X_tr[..., np.newaxis]
X_val  = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

cw = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_tr)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}

print("train:", X_tr.shape, " val:", X_val.shape, " test:", X_test.shape)
print("class weights:", {SHORT[k]: round(v,2) for k,v in CLASS_WEIGHT.items()})

---
## 4. Shared training + evaluation harness

Identical budget for every architecture — same optimiser, same epochs, same callbacks,
same class weights. The only variable is the architecture. That is what makes this a
**controlled comparison** rather than a leaderboard, and it is the point to make in the viva.

In [ ]:
EPOCHS = 40
RESULTS = {}
HISTORIES = {}

def make_callbacks(tag):
    return [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=6,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=3, min_lr=1e-5, verbose=1),
        keras.callbacks.ModelCheckpoint(f'/content/{tag}_best.keras',
                                        monitor='val_loss', save_best_only=True),
    ]

def run(model, tag, batch_size=128):
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    n_params = model.count_params()
    print(f"\n{'='*70}\n{tag}  |  {n_params:,} parameters\n{'='*70}")
    model.summary()

    t0 = time.time()
    hist = model.fit(X_tr, y_tr,
                     validation_data=(X_val, y_val),
                     epochs=EPOCHS, batch_size=batch_size,
                     class_weight=CLASS_WEIGHT,
                     callbacks=make_callbacks(tag), verbose=2)
    train_time = time.time() - t0

    t1 = time.time()
    proba = model.predict(X_test, batch_size=512, verbose=0)
    infer_ms = 1000 * (time.time() - t1) / len(X_test)
    pred = proba.argmax(1)

    prec, rec, f1, sup = precision_recall_fscore_support(
        y_test, pred, labels=range(N_CLASSES), zero_division=0)

    RESULTS[tag] = {
        'params': int(n_params),
        'epochs_run': len(hist.history['loss']),
        'train_time_s': round(train_time, 1),
        'inference_ms_per_beat': round(infer_ms, 4),
        'accuracy': round(accuracy_score(y_test, pred), 4),
        'macro_f1': round(f1_score(y_test, pred, average='macro'), 4),
        'weighted_f1': round(f1_score(y_test, pred, average='weighted'), 4),
        'macro_auc_ovr': round(roc_auc_score(y_test, proba, multi_class='ovr',
                                             average='macro'), 4),
        'per_class_recall': {SHORT[i]: round(float(rec[i]), 4) for i in range(N_CLASSES)},
        'per_class_f1':     {SHORT[i]: round(float(f1[i]),  4) for i in range(N_CLASSES)},
    }
    HISTORIES[tag] = hist.history
    np.save(f'/content/{tag}_pred.npy', pred)

    print(f"\n--- {tag} test results ---")
    print(f"accuracy   {RESULTS[tag]['accuracy']:.4f}")
    print(f"macro F1   {RESULTS[tag]['macro_f1']:.4f}   <-- headline")
    print(f"macro AUC  {RESULTS[tag]['macro_auc_ovr']:.4f}")
    print(f"train time {train_time:.1f}s over {len(hist.history['loss'])} epochs\n")
    print(classification_report(y_test, pred, target_names=CLASS_NAMES, digits=4))
    return model, pred

---
## 5. Model A — 1D CNN

Four convolutional blocks with increasing filter counts, batch normalisation for training
stability, and **global average pooling** instead of flattening (far fewer parameters,
less overfitting — the tabular ANN in the previous phase overfit badly, so this matters).

Kernel size 5 at 125 Hz covers a 40 ms window, roughly the width of a QRS complex.

In [ ]:
def build_cnn():
    m = keras.Sequential(name='CNN_1D')
    m.add(layers.Input(shape=(187, 1)))
    for filters in [32, 64, 128, 128]:
        m.add(layers.Conv1D(filters, 5, padding='same', activation='relu'))
        m.add(layers.BatchNormalization())
        m.add(layers.Conv1D(filters, 5, padding='same', activation='relu'))
        m.add(layers.BatchNormalization())
        m.add(layers.MaxPooling1D(2))
        m.add(layers.Dropout(0.2))
    m.add(layers.GlobalAveragePooling1D())
    m.add(layers.Dense(128, activation='relu'))
    m.add(layers.Dropout(0.4))
    m.add(layers.Dense(N_CLASSES, activation='softmax'))
    return m

cnn_model, cnn_pred = run(build_cnn(), 'CNN', batch_size=128)

---
## 6. Model B — Bidirectional LSTM

Two stacked BiLSTM layers. **Deliberately left with default activations and
`recurrent_dropout=0`** so Keras dispatches to the fused cuDNN kernel — setting
recurrent dropout silently falls back to a generic implementation that is roughly
10× slower on a T4. Worth a sentence in the report as an engineering decision.

Larger batch size (256) because recurrent layers are sequential and latency-bound.

In [ ]:
def build_lstm():
    m = keras.Sequential(name='BiLSTM')
    m.add(layers.Input(shape=(187, 1)))
    m.add(layers.Bidirectional(layers.LSTM(64, return_sequences=True)))
    m.add(layers.Dropout(0.3))
    m.add(layers.Bidirectional(layers.LSTM(64)))
    m.add(layers.Dropout(0.3))
    m.add(layers.Dense(64, activation='relu'))
    m.add(layers.Dropout(0.3))
    m.add(layers.Dense(N_CLASSES, activation='softmax'))
    return m

lstm_model, lstm_pred = run(build_lstm(), 'BiLSTM', batch_size=256)

---
## 7. Model C — CNN-BiLSTM hybrid

Convolutional front-end extracts local morphology and downsamples 187 → ~46 timesteps,
then the BiLSTM models temporal relationships over those learned features. The shorter
sequence makes the recurrent layer far cheaper than in Model B.

This is the architecture most current ECG papers converge on, so including it means you
can say in the viva that you tested the state of the art rather than only the textbook baselines.

In [ ]:
def build_hybrid():
    m = keras.Sequential(name='CNN_BiLSTM')
    m.add(layers.Input(shape=(187, 1)))
    for filters in [32, 64]:
        m.add(layers.Conv1D(filters, 5, padding='same', activation='relu'))
        m.add(layers.BatchNormalization())
        m.add(layers.MaxPooling1D(2))
    m.add(layers.Dropout(0.2))
    m.add(layers.Bidirectional(layers.LSTM(64, return_sequences=True)))
    m.add(layers.Bidirectional(layers.LSTM(64)))
    m.add(layers.Dropout(0.3))
    m.add(layers.Dense(64, activation='relu'))
    m.add(layers.Dropout(0.3))
    m.add(layers.Dense(N_CLASSES, activation='softmax'))
    return m

hybrid_model, hybrid_pred = run(build_hybrid(), 'CNN_BiLSTM', batch_size=256)

---
## 8. Comparison

In [ ]:
comp = pd.DataFrame(RESULTS).T[
    ['accuracy','macro_f1','weighted_f1','macro_auc_ovr',
     'params','epochs_run','train_time_s','inference_ms_per_beat']]
comp = comp.sort_values('macro_f1', ascending=False)
display(comp)

winner = comp.index[0]
print(f"\nBest by macro F1: {winner}  ({comp.loc[winner,'macro_f1']:.4f})")

In [ ]:
recall_tbl = pd.DataFrame({k: v['per_class_recall'] for k, v in RESULTS.items()}).T
print("Per-class RECALL (sensitivity) — the clinically important view:")
display(recall_tbl.style.background_gradient(cmap='RdYlGn', axis=None, vmin=0.5, vmax=1.0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for tag, h in HISTORIES.items():
    axes[0].plot(h['val_loss'], label=tag, lw=2)
    axes[1].plot(h['val_accuracy'], label=tag, lw=2)
axes[0].set_title('Validation loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].set_title('Validation accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout(); plt.savefig('/content/fig_curves.png', bbox_inches='tight'); plt.show()

In [ ]:
preds = {'CNN': cnn_pred, 'BiLSTM': lstm_pred, 'CNN_BiLSTM': hybrid_pred}
fig, axes = plt.subplots(1, len(preds), figsize=(6*len(preds), 5))
for ax, (tag, p) in zip(np.atleast_1d(axes), preds.items()):
    cm = confusion_matrix(y_test, p, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1,
                xticklabels=SHORT, yticklabels=SHORT, ax=ax, cbar=False)
    ax.set_title(f"{tag} — macro F1 {RESULTS[tag]['macro_f1']:.4f}")
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
plt.suptitle('Row-normalised confusion matrices (MIT-BIH test set)', y=1.03, fontsize=13)
plt.tight_layout(); plt.savefig('/content/fig_confusion.png', bbox_inches='tight'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for tag, r in RESULTS.items():
    ax.scatter(r['train_time_s'], r['macro_f1'], s=np.sqrt(r['params'])/2.5,
               alpha=0.65, label=f"{tag} ({r['params']:,} params)")
    ax.annotate(tag, (r['train_time_s'], r['macro_f1']),
                textcoords='offset points', xytext=(8, 8), fontsize=10)
ax.set_xlabel('training time (s)'); ax.set_ylabel('macro F1 (test)')
ax.set_title('Accuracy vs computational cost\n(bubble area ∝ parameter count)')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.savefig('/content/fig_cost.png', bbox_inches='tight'); plt.show()

---
## 9. Persist results

Saves a JSON of every metric plus the trained models, so the report and the FastAPI
backend can consume them without retraining.

In [ ]:
out = {
    'phase': 'CNN vs LSTM — MIT-BIH ECG arrhythmia',
    'dataset': 'shayanfazeli/heartbeat (MIT-BIH, 187-sample beats, 5 AAMI classes)',
    'n_train': int(len(X_tr)), 'n_val': int(len(X_val)), 'n_test': int(len(X_test)),
    'headline_metric': 'macro_f1',
    'class_weighting': 'balanced',
    'seed': SEED,
    'results': RESULTS,
}
with open('/content/ecg_results.json', 'w') as f:
    json.dump(out, f, indent=2)

print(json.dumps(out['results'], indent=2))

from google.colab import files as gfiles
!zip -q -r /content/ecg_phase_outputs.zip /content/ecg_results.json /content/fig_*.png /content/*_best.keras
gfiles.download('/content/ecg_phase_outputs.zip')

---
## 10. Report + viva notes

### The result you will most likely get
The CNN should beat the BiLSTM on macro F1 while training several times faster. That is a
**good** result, not a disappointing one — it means your architecture choice is *explainable*:
a single 187-sample beat is a morphology problem, and convolution encodes the right prior.
The LSTM's sequence-modelling capacity has little to exploit inside one isolated beat.

Pair this with the tabular finding and you have a genuine, defensible narrative:

> *"Model performance is not a fixed ranking — it is a function of data structure.
> Tree ensembles and pretrained transformers won on small tabular clinical records;
> convolutional networks won on ECG waveforms. Selecting a model without characterising
> the data first is the actual mistake."*

That single claim covers supervisor points (2), (4), (7) and (9) at once.

---

### ⚠️ Read this before you write the results chapter

This Kaggle release splits beats **randomly**, so heartbeats from the *same patient* appear
in both train and test. Under the inter-patient evaluation protocol of de Chazal et al. (2004),
where whole patient records are held out, reported accuracy on MIT-BIH typically drops from
~98% to the low 80s.

**This is not a flaw in your work — it is your best material.** It is the concrete, evidenced
answer to IPR marker point (5) and supervisor point (10): *does strong benchmark performance
imply real-world clinical reliability?* Here you can demonstrate that it does not, with numbers,
on your own artefact. Almost no undergraduate report does this. State the limitation explicitly,
explain the mechanism (patient-specific morphology leakage), and cite de Chazal.

**Optional but high-value:** re-run the CNN on a patient-disjoint split from the raw `wfdb`
records and report both figures side by side. That single extra table is probably worth more
marks than any further architecture tuning.

---

### Success criteria to state up front (IPR marker point 1)
Declare these *before* presenting results, so the evaluation reads as pre-registered:

- Macro F1 ≥ 0.85 on the held-out test set
- Recall ≥ 0.90 on class V (ventricular ectopic — the highest-acuity class)
- Inference < 10 ms per beat, so the FastAPI endpoint stays interactive

---

### Integration into HealthGuard AI
Keep the ECG model as a **separate endpoint**, not fused into the tabular risk score. The two
consume different inputs and answer different questions; merging them would force a single
confidence figure that misrepresents both. Say that explicitly — it is exactly the kind of
"decision rejected and why" evidence the IPR marker asked for (point 4).

### Reference
de Chazal, P., O'Dwyer, M. and Reilly, R.B. (2004) 'Automatic classification of heartbeats
using ECG morphology and heartbeat interval features', *IEEE Transactions on Biomedical
Engineering*, 51(7), pp. 1196–1206.